In [1]:
import os
import pandas as pd
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm
import numpy as np

# Configuração de caminhos
BASE_DIR = Path("/home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios")
IBAMA_BRONZE_DIR = BASE_DIR / "ibama/data/bronze/ibama_embargos"
OUTPUT_DIR = BASE_DIR / "ibama/data/resume"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def analyze_file(file_path):
    name = file_path.name
    print(f"Analisando arquivo: {name}")
    try:
        # 1. Correção na leitura: Prioriza geopandas para geoparquet
        if name.endswith(".geoparquet"):
            df = gpd.read_parquet(file_path)
        else:
            # Tenta ler com pandas, mas se falhar por ser geoparquet disfarçado, usa geopandas
            try:
                df = pd.read_parquet(file_path)
            except:
                df = gpd.read_parquet(file_path)
            
        num_rows = len(df)
        if num_rows == 0:
            print(f"  ⚠️ Arquivo {name} está vazio.")
            return None

        # Memória em MB
        mem_usage = df.memory_usage(deep=True).sum() / (1024**2)
        
        summary = []
        for col in df.columns:
            series = df[col]
            null_count = int(series.isna().sum())
            dtype_str = str(series.dtype)
            
            stats = {
                "file": name,
                "total_rows": num_rows,
                "mem_mb": round(mem_usage, 2),
                "column": col,
                "dtype": dtype_str,
                "null_count": null_count,
                "null_percent": round((null_count / num_rows) * 100, 2),
                "unique_values": 0,
                "min": "-",
                "max": "-",
                "sample_values": "[]"
            }

            # 2. Cálculo seguro de valores únicos (evita erro em tipos não hashable)
            try:
                stats["unique_values"] = int(series.nunique())
                sample = series.dropna().unique()[:3].tolist()
                stats["sample_values"] = str(sample)
            except:
                stats["sample_values"] = "Erro ao extrair amostra"

            # 3. Min/Max Seguro: Pula colunas de geometria e lida com erros de comparação
            if "geometry" not in dtype_str.lower():
                try:
                    if np.issubdtype(series.dtype, np.number) or np.issubdtype(series.dtype, np.datetime64):
                        stats["min"] = str(series.min())
                        stats["max"] = str(series.max())
                except:
                    pass
                
            summary.append(stats)
            
        return summary
    except Exception as e:
        print(f"  ❌ Erro crítico ao analisar {name}: {str(e)}")
        return None

# Busca arquivos (Case-insensitive para extensões)
files_to_analyze = []
for ext in ["*.parquet", "*.PARQUET", "*.geoparquet", "*.GEOPARQUET"]:
    files_to_analyze.extend(list(IBAMA_BRONZE_DIR.glob(ext)))

all_summaries = []
for file_path in tqdm(files_to_analyze, desc="Progresso"):
    file_summary = analyze_file(file_path)
    if file_summary:
        all_summaries.extend(file_summary)

if all_summaries:
    summary_df = pd.DataFrame(all_summaries)
    output_file = OUTPUT_DIR / "resumo_detalhado_ibama.csv"
    summary_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n✅ Resumo exportado com sucesso para: {output_file}")
    display(summary_df.head(10))
else:
    print("\n❌ Nenhum dado foi processado. Verifique se os caminhos estão corretos.")

Progresso:   0%|          | 0/2 [00:00<?, ?it/s]

Analisando arquivo: embargos_ibama_tabular.parquet


Progresso:  50%|█████     | 1/2 [00:00<00:00,  2.97it/s]

Analisando arquivo: embargos_ibama_full.geoparquet


Progresso: 100%|██████████| 2/2 [00:01<00:00,  1.30it/s]


✅ Resumo exportado com sucesso para: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/ibama/data/resume/resumo_detalhado_ibama.csv


,file,total_rows,mem_mb,column,dtype,null_count,null_percent,unique_values,min,max,sample_values
0,embargos_ibama_tabular.parquet,88586,72.39,objectid,int64,0,0.00,88586,1,88586,"[25298, 25299, 25300]"
1,embargos_ibama_tabular.parquet,88586,72.39,seq_tad,int64,0,0.00,87533,0,1876749,"[1639007, 1472789, 1610984]"
2,embargos_ibama_tabular.parquet,88586,72.39,num_tad,str,0,0.00,87698,-,-,"['756611', '602347', '729878']"
3,embargos_ibama_tabular.parquet,88586,72.39,serie_tad,str,24615,27.79,5,-,-,"['E', 'C', 'A']"
4,embargos_ibama_tabular.parquet,88586,72.39,operacao,str,79121,89.32,747,-,-,"['ONDA VERDE P11', 'ONDA VERDE', 'CONTROLE REM..."
5,embargos_ibama_tabular.parquet,88586,72.39,origem_geo,str,0,0.00,3,-,-,"['Polígono', 'Sem Geometria', 'Ponto']"
6,embargos_ibama_tabular.parquet,88586,72.39,cod_uf,str,0,0.00,27,-,-,"['13', '29', '15']"
7,embargos_ibama_tabular.parquet,88586,72.39,uf,str,0,0.00,27,-,-,"['AM', 'BA', 'PA']"
8,embargos_ibama_tabular.parquet,88586,72.39,cod_munici,int64,0,0.00,3769,1100015,9999999,"[1302405, 2927705, 1500602]"
9,embargos_ibama_tabular.parquet,88586,72.39,municipio,str,0,0.00,3633,-,-,"['Lábrea', 'Santa Cruz Cabrália', 'Altamira']"
